# Import Modules

In [1]:
import math
import random
import folium
import requests
import numpy as np
import gravis as gv
import pandas as pd
import networkx as nx

from tqdm import tqdm

c:\Users\halko\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\gravis\_internal\plotting\template_system.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources as _pkg_resources


# Load data

In [2]:
# Load data
stops_df = pd.read_csv("timetables-20260312\\var\\www\\data.datalibrary.uk\\transport\\BODS-ARCHIVE\\timetables\\2026\\03\\12\\itm_yorkshire_gtfs_20260312\\stops.txt")
stop_times_df = pd.read_csv("timetables-20260312\\var\\www\\data.datalibrary.uk\\transport\\BODS-ARCHIVE\\timetables\\2026\\03\\12\\itm_yorkshire_gtfs_20260312\\stop_times.txt")
trips = pd.read_csv("timetables-20260312\\var\\www\\data.datalibrary.uk\\transport\\BODS-ARCHIVE\\timetables\\2026\\03\\12\\itm_yorkshire_gtfs_20260312\\trips.txt")

# Convert datatypes
stop_times_df["arrival_time"] = pd.to_timedelta(stop_times_df["arrival_time"])
stop_times_df["departure_time"] = pd.to_timedelta(stop_times_df["departure_time"])

# Merge route id details into stop times
stop_times_df = stop_times_df.merge(
    trips[["route_id", "trip_id"]],
    left_on='trip_id', 
    right_on='trip_id', 
    how='left')

# # Isolate Bradford
# stops_df = stops_df.loc[(stops_df["stop_lat"] < 54) & (stops_df["stop_lat"] > 53.7)].reset_index(drop=True)
# stops_df = stops_df.loc[(stops_df["stop_lon"] < -1.55) & (stops_df["stop_lon"] > -1.95)].reset_index(drop=True)

In [3]:
# docker run -t -i -p 5001:5000 -v "${PWD}:/data" osrm/osrm-backend osrm-routed --algorithm mld /data/west-yorkshire-foot.osrm

def time_journey(origin, dest):
    o_str = f"{origin[1]},{origin[0]}"
    d_str = f"{dest[1]},{dest[0]}"
    
    url = f"http://127.0.0.1:5001/route/v1/foot/{o_str};{d_str}?overview=false"
    
    response = requests.get(url).json()
    return response['routes'][0]['duration']

In [4]:
# trip_stops = set()
# unique_trips = set()

# dupe_count = 0
# for trip in stop_times_df["trip_id"].unique():
#     # If this trip is just a subset of an existing trip then skip it
#     # Otherwise, if this trip is 
#     stops_in_trip = tuple(stop_times_df.loc[stop_times_df["trip_id"] == trip, "stop_id"].unique())
#     if stops_in_trip in trip_stops:
#         dupe_count += 1
#         print("Duplicate trip", dupe_count)
#         continue
#     else:
#         trip_stops.add(stops_in_trip)
#         unique_trips.add(trip)


In [5]:
grouped = stop_times_df.groupby("trip_id")["stop_sequence"]
stop_times_df["trip_len"] = (grouped.transform("max") - grouped.transform("min")) + 1

longest_trip_ids = (
    stop_times_df
    .sort_values("trip_len", ascending=False)
    .drop_duplicates(["route_id"])["trip_id"]
)

longest_trips_df = stop_times_df[stop_times_df["trip_id"].isin(longest_trip_ids)].reset_index(drop=True)

In [6]:
stop_times_df.loc[[5756462, 6199129]]

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint,route_id,trip_len
5756462,VJc514c05adf9f1f7f1080c1f7a6764fc76c04881b,0 days 17:42:00,0 days 17:42:00,3200YNA97209,20,NaN,0,0,NaN,0,1283259,181
6199129,VJd4488eafd8336fffd2ef95bfc9822e06a1073bf2,0 days 15:05:00,0 days 15:05:00,450011613,147,NaN,0,0,NaN,0,101221,177


In [7]:
trip_stops_dict = stop_times_df.groupby('trip_id')['stop_id'].apply(set).to_dict()

# 2. Extract trip-level information to reduce dimensionality
# We drop from millions of stop-time rows down to just the unique trips
trip_info = stop_times_df[['route_id', 'trip_id', 'stop_sequence', 'trip_len']].drop_duplicates()

# Sort descending once so that when we group, the longest trip is always first
trip_info = trip_info.sort_values(by=['route_id', 'trip_len'], ascending=[True, False])

In [8]:
include_ids = []
considered_routes = set()

# 3. Iterate over grouped trips rather than the full stop_times DataFrame
for route_id, route_trips in tqdm(trip_info.groupby('route_id')):
    if route_id in considered_routes:
        print(f"Already consided route {route_id}, idk how it's ended up here again...")
    considered_routes.add(route_id)
    # Skip if route has no trips (edge case)
    if route_trips.empty:
        continue
        
    # Because we sorted earlier, the longest trip is guaranteed to be the first row
    longest_trip_id = route_trips.iloc[0]['trip_id']
    include_ids.append(longest_trip_id)
    
    # Fetch pre-computed stops for the longest trip
    longest_trip_stops = list(trip_stops_dict[longest_trip_id])
    
    # Get two random stops (handle case where trip has < 2 stops)
    sample_stops = set(np.random.choice(longest_trip_stops, 2, replace=False))
        
    # Find the next longest trip not containing these stops
    # We slice [1:] to skip the longest trip we already evaluated
    for trip_id in route_trips['trip_id'].iloc[1:]:
        cur_stops = trip_stops_dict[trip_id]
        
        # isdisjoint() is highly optimized in C for Python sets
        if cur_stops.isdisjoint(sample_stops):
            include_ids.append(trip_id)
            break

# 4. Final single filter step
longest_trips_df = stop_times_df[stop_times_df["trip_id"].isin(include_ids)].reset_index(drop=True)

100%|██████████| 1140/1140 [00:01<00:00, 813.68it/s] 


In [9]:
longest_trips_df

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint,route_id,trip_len
0,VJ00004638078b95f09f34abf1dcfb3c6c70413692,0 days 14:15:00,0 days 14:15:00,450029815,0,NaN,0,1,NaN,1,50628,82
1,VJ00004638078b95f09f34abf1dcfb3c6c70413692,0 days 14:18:00,0 days 14:18:00,450032483,1,NaN,0,0,NaN,0,50628,82
2,VJ00004638078b95f09f34abf1dcfb3c6c70413692,0 days 14:19:00,0 days 14:19:00,450032481,2,NaN,0,0,NaN,0,50628,82
3,VJ00004638078b95f09f34abf1dcfb3c6c70413692,0 days 14:21:00,0 days 14:21:00,450025175,3,NaN,0,0,NaN,0,50628,82
4,VJ00004638078b95f09f34abf1dcfb3c6c70413692,0 days 14:22:00,0 days 14:22:00,450024824,4,NaN,0,0,NaN,0,50628,82
...,...,...,...,...,...,...,...,...,...,...,...,...
84699,VJfef2fbf12ca67090a043f74adbaadd5a3d2f69be,0 days 08:47:00,0 days 08:47:00,450018435,64,NaN,0,0,NaN,0,11930,69
84700,VJfef2fbf12ca67090a043f74adbaadd5a3d2f69be,0 days 08:49:00,0 days 08:49:00,450018436,65,NaN,0,0,NaN,0,11930,69
84701,VJfef2fbf12ca67090a043f74adbaadd5a3d2f69be,0 days 08:51:00,0 days 08:51:00,450026346,66,NaN,0,0,NaN,0,11930,69
84702,VJfef2fbf12ca67090a043f74adbaadd5a3d2f69be,0 days 08:52:00,0 days 08:52:00,450025213,67,NaN,0,0,NaN,0,11930,69


In [10]:
# stop_times_df.sort_values("trip_len", ascending=False)

# include_ids = []
# # for each route id
# # find longest trip
# # get two stops along that trip
# # find the longest trip for this route which does NOT include those two stops
# # if one does exists then add index to include list

# # for each route id
# for route_id in tqdm(stop_times_df["route_id"].unique()):
#     route_stops_df = stop_times_df.loc[stop_times_df["route_id"] == route_id].sort_values("trip_len", ascending=False)

#     # find longest trip
#     long_trip_index = route_stops_df["trip_len"].idxmax()
#     long_trip_id = stop_times_df.loc[long_trip_index, "trip_id"]

#     # include longest trip
#     include_ids.append(long_trip_id)

#     # get two stops along that trip
#     long_trip = stop_times_df.loc[stop_times_df["trip_id"] == long_trip_id].reset_index()
#     sample_stops = long_trip.sample(2)["stop_id"].to_list()
    
#     # find the longest trip for this route which does NOT include those two stops
#     for trip_id in route_stops_df["trip_id"].unique():
#         cur_trip = route_stops_df.loc[route_stops_df["trip_id"] == trip_id]
#         cur_trip_stops = cur_trip["stop_id"].values

#         if not any(stop in cur_trip_stops for stop in sample_stops):
#             include_ids.append(trip_id)
#             break

# stop_times_df[stop_times_df["trip_id"].isin(include_ids)].reset_index(drop=True)

# Debug weirdness

In [11]:
route_id = 1283259
f"Longest single trip on route {route_id}: {int(stop_times_df.loc[stop_times_df["route_id"] == route_id, "trip_len"].max())}; Total number of unique stops with that route id: {len(stop_times_df.loc[stop_times_df["route_id"] == route_id, "stop_id"].unique())}"

'Longest single trip on route 1283259: 181; Total number of unique stops with that route id: 292'

In [12]:
stop_times_df.loc[stop_times_df["route_id"] == route_id]

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint,route_id,trip_len
2290995,VJ4e10363251bea28222b6dcbbe04ccc5803d6e58d,0 days 08:05:00,0 days 08:05:00,3290YYA00698,0,NaN,0,1,NaN,1,1283259,118
2290996,VJ4e10363251bea28222b6dcbbe04ccc5803d6e58d,0 days 08:06:00,0 days 08:06:00,3290YYA00700,1,NaN,0,0,NaN,0,1283259,118
2290997,VJ4e10363251bea28222b6dcbbe04ccc5803d6e58d,0 days 08:07:00,0 days 08:07:00,3290YYA00002,2,NaN,0,0,NaN,0,1283259,118
2290998,VJ4e10363251bea28222b6dcbbe04ccc5803d6e58d,0 days 08:08:00,0 days 08:08:00,3290YYA00384,3,NaN,0,0,NaN,0,1283259,118
2290999,VJ4e10363251bea28222b6dcbbe04ccc5803d6e58d,0 days 08:09:00,0 days 08:09:00,3290YYA00382,4,NaN,0,0,NaN,0,1283259,118
...,...,...,...,...,...,...,...,...,...,...,...,...
5756618,VJc514c05adf9f1f7f1080c1f7a6764fc76c04881b,0 days 20:22:00,0 days 20:22:00,3290YYA00381,176,NaN,0,0,NaN,0,1283259,181
5756619,VJc514c05adf9f1f7f1080c1f7a6764fc76c04881b,0 days 20:23:00,0 days 20:23:00,3290YYA00383,177,NaN,0,0,NaN,0,1283259,181
5756620,VJc514c05adf9f1f7f1080c1f7a6764fc76c04881b,0 days 20:24:00,0 days 20:24:00,3290YYA00001,178,NaN,0,0,NaN,0,1283259,181
5756621,VJc514c05adf9f1f7f1080c1f7a6764fc76c04881b,0 days 20:26:00,0 days 20:26:00,3290YYA00701,179,NaN,0,0,NaN,0,1283259,181


In [16]:
long_route = longest_trips_df.loc[longest_trips_df["route_id"] == route_id].reset_index()
# long_route = selected_trips_df.loc[selected_trips_df["route_id"] == route_id].reset_index()

all_stops = stop_times_df.loc[stop_times_df["route_id"] == route_id, "stop_id"].unique()

In [17]:
first_stop = long_route.loc[0, "stop_id"]
start_lat = stops_df.loc[stops_df["stop_id"] == first_stop, "stop_lat"].item()
start_lon = stops_df.loc[stops_df["stop_id"] == first_stop, "stop_lon"].item()

last_stop = long_route.loc[len(long_route)-2, "stop_id"]
end_lat = stops_df.loc[stops_df["stop_id"] == last_stop, "stop_lat"].item()
end_lon = stops_df.loc[stops_df["stop_id"] == last_stop, "stop_lon"].item()

print(end_lat, end_lon)

map = folium.Map(location=[(start_lat+end_lat)/2, (start_lon+end_lon)/2], zoom_start=10,  tiles='CartoDB positron')

for stop in all_stops:
    lat = stops_df.loc[stops_df["stop_id"] == stop, "stop_lat"].item()
    lon = stops_df.loc[stops_df["stop_id"] == stop, "stop_lon"].item()

    folium.CircleMarker(
        location=(lat, lon),
        radius=4,
        color="red",
        fill=True,
        fill_color="white",
        fill_opacity=0.5,
        weight=3,
        # tooltip=f"Stop: {stop['stop_id']}"
    ).add_to(map)

for stop in long_route["stop_id"].unique():
    lat = stops_df.loc[stops_df["stop_id"] == stop, "stop_lat"].item()
    lon = stops_df.loc[stops_df["stop_id"] == stop, "stop_lon"].item()

    folium.CircleMarker(
        location=(lat, lon),
        radius=4,
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.5,
        weight=0,
        # tooltip=f"Stop: {stop['stop_id']}"
    ).add_to(map)

map.save("debug_route_stop_map.html")

53.983882096 -1.119062655


# ROUTES GO BOTH WAYS LOL

# Create basic graph

In [18]:
G = nx.MultiGraph()

longest_trips_df = longest_trips_df.sort_values(["trip_id", "stop_sequence"])

# for route_id in longest_trips_df["route_id"].unique():
#     trip_to_plot = longest_trips_df.loc[longest_trips_df["route_id"] == route_id].reset_index()

#     for i in range(len(trip_to_plot) - 1):
#         cur_stop = trip_to_plot.loc[i, "stop_id"]
#         next_stop = trip_to_plot.loc[i+1, "stop_id"]

#         # cur_stop_lat = stops_df.loc[stops_df["stop_id"] == cur_stop, "stop_lat"].item()
#         # cur_stop_lon = stops_df.loc[stops_df["stop_id"] == cur_stop, "stop_lon"].item()

#         duration = trip_to_plot.loc[i+1, "arrival_time"] - trip_to_plot.loc[i, "departure_time"]
#         duration_sec = duration.total_seconds()
#         G.add_edge(cur_stop, next_stop, weight=duration_sec)

for trip_id, trip_to_plot in longest_trips_df.groupby("trip_id"):
    trip_to_plot = trip_to_plot.reset_index(drop=True)

    for i in range(len(trip_to_plot) - 1):
        cur_stop = trip_to_plot.loc[i, "stop_id"]
        next_stop = trip_to_plot.loc[i+1, "stop_id"]

        duration = trip_to_plot.loc[i+1, "arrival_time"] - trip_to_plot.loc[i, "departure_time"]
        duration_sec = duration.total_seconds()
        
        # Guard against bad GTFS data or negative time jumps
        if duration_sec >= 0:
            G.add_edge(cur_stop, next_stop, weight=duration_sec)
        else:
            # Handle overnight trips or data anomalies if necessary
            print(f"Warning: Negative duration found on trip {trip_id}")

# 2. Add node attributes in bulk
# Set stop_id as the index and convert the dataframe to a nested dictionary
stops_dict = stops_df.set_index("stop_id")[["stop_lat", "stop_lon"]].to_dict("index")

# Rename the keys to match the 'lat' and 'lon' expected by the heuristic
node_attributes = {
    stop_id: {"lat": data["stop_lat"], "lon": data["stop_lon"]}
    for stop_id, data in stops_dict.items() 
    if stop_id in G.nodes # Only add attributes for nodes that exist in the graph
}

nx.set_node_attributes(G, node_attributes)

## Search Graph

In [19]:
MEAN_LAT_RAD = math.radians(53.8) 
LON_SCALE = math.cos(MEAN_LAT_RAD)

METERS_PER_DEGREE = 111320.0
MAX_SPEED_MPS = 25.0

def manhattan_heuristic(node1, node2):
    lat1, lon1 = G.nodes[node1]['lat'], G.nodes[node1]['lon']
    lat2, lon2 = G.nodes[node2]['lat'], G.nodes[node2]['lon']
    
    dlat = abs(lat2 - lat1)
    dlon = abs(lon2 - lon1) * LON_SCALE
    
    # distance = METERS_PER_DEGREE * (dlat + dlon)
    
    return (dlat + dlon)# distance / MAX_SPEED_MPS

In [20]:
start_stop_id = "450014118"
end_stop_id = "450021125"

try:
    optimal_path = nx.astar_path(
        G=G,
        source=start_stop_id,
        target=end_stop_id,
        heuristic=manhattan_heuristic,
        weight="weight"
    )
    print(f"Path: {optimal_path}")
    
    optimal_duration = nx.astar_path_length(
        G=G,
        source=start_stop_id,
        target=end_stop_id,
        heuristic=manhattan_heuristic,
        weight="weight"
    )
    
    print(f"Total Duration: {optimal_duration//60} mins")

except nx.NetworkXNoPath:
    print("No path exists between the specified stops.")
except nx.NodeNotFound as e:
    print(f"Node missing from graph: {e}")

Path: ['450014118', '450019715', '450024198', '450014117', '450014119', '450018826', '450018824', '450018821', '450018815', '450018837', '450018840', '450018841', '450024303', '450018842', '450018844', '450018846', '450018851', '450018852', '450018854', '450018857', '450018858', '450027448', '450018861', '450024365', '450018644', '450018642', '450018649', '450018650', '450020527', '450020531', '450020532', '450020535', '450020537', '450021121', '450021122', '450021123', '450021124', '450021125']
Total Duration: 12.0 mins


## Draw graph

In [21]:
start_node = optimal_path[0]
start_lat, start_lon = G.nodes[start_node]['lat'], G.nodes[start_node]['lon']
route_map = folium.Map(location=[start_lat, start_lon], zoom_start=12,  tiles='CartoDB positron')

folium.Marker(
    location=(start_lat, start_lon),
    popup="Start",
    icon=folium.Icon(color="green", icon="play")
).add_to(route_map)

end_node = optimal_path[-1]
folium.Marker(
    location=(G.nodes[end_node]['lat'], G.nodes[end_node]['lon']),
    popup="Destination",
    icon=folium.Icon(color="red", icon="stop")
).add_to(route_map)

for node in optimal_path:
    folium.CircleMarker(
        location=(G.nodes[node]["lat"], G.nodes[node]["lon"]),
        radius=4,
        # color=color,
        fill=True,
        fill_color="white",
        fill_opacity=1,
        weight=2,
        # tooltip=f"Stop: {stop['stop_id']}"
    ).add_to(route_map)

route_map.save("a_star_optimal_route.html")

# Tranfer cost graph

In [34]:
# 1. Use a Directed Graph. This is strictly required so that 
# passengers cannot use 0-cost alighting edges backwards to board.
G = nx.MultiDiGraph() 

TRANSFER_PENALTY_SEC = 1200  # 10 minute penalty for boarding a new route

# 2. Ensure data is strictly ordered by trip and sequence
longest_trips_df = longest_trips_df.sort_values(["trip_id", "stop_sequence"])

# 3. Group by TRIP, not route
for trip_id, trip_to_plot in longest_trips_df.groupby("trip_id"):
    trip_to_plot = trip_to_plot.reset_index(drop=True)
    
    # Extract the route_id for this specific trip to label the platforms
    route_id = trip_to_plot["route_id"].iloc[0]

    for i in range(len(trip_to_plot) - 1):
        cur_stop = trip_to_plot.loc[i, "stop_id"]
        next_stop = trip_to_plot.loc[i+1, "stop_id"]

        # Define platform nodes as tuples
        cur_platform = (cur_stop, route_id)
        next_platform = (next_stop, route_id)

        duration = trip_to_plot.loc[i+1, "arrival_time"] - trip_to_plot.loc[i, "departure_time"]
        duration_sec = duration.total_seconds()
        
        # Guard against negative time loops
        if duration_sec < 0:
            continue

        # --- 1. Travel Edge (Platform to Platform) ---
        # Directed edge: Train only goes one way
        G.add_edge(cur_platform, next_platform, weight=duration_sec)
        
        # --- 2. Boarding Edges (Station -> Platform) ---
        # Directed edge: Station to Platform costs the penalty
        if not G.has_edge(cur_stop, cur_platform):
            G.add_edge(cur_stop, cur_platform, weight=TRANSFER_PENALTY_SEC)
            
        if not G.has_edge(next_stop, next_platform):
            G.add_edge(next_stop, next_platform, weight=TRANSFER_PENALTY_SEC)
            
        # --- 3. Alighting Edges (Platform -> Station) ---
        # Directed edge: Platform to Station is free
        if not G.has_edge(cur_platform, cur_stop):
            G.add_edge(cur_platform, cur_stop, weight=0)
            
        if not G.has_edge(next_platform, next_stop):
            G.add_edge(next_platform, next_stop, weight=0)

In [35]:
stops_dict = stops_df.set_index("stop_id")[["stop_lat", "stop_lon"]].to_dict("index")

node_attributes = {}

for node in G.nodes:
    if isinstance(node, tuple):
        # This is a Platform node: (stop_id, route_id)
        parent_stop_id = node[0]
        if parent_stop_id in stops_dict:
            node_attributes[node] = {
                "lat": stops_dict[parent_stop_id]["stop_lat"], 
                "lon": stops_dict[parent_stop_id]["stop_lon"]
            }
    else:
        # This is a Station node: stop_id
        if node in stops_dict:
            node_attributes[node] = {
                "lat": stops_dict[node]["stop_lat"], 
                "lon": stops_dict[node]["stop_lon"]
            }

nx.set_node_attributes(G, node_attributes)

## Search Graph Again

In [36]:
start_stop_id = "450014118"
end_stop_id = "450021125"

try:
    optimal_path = nx.astar_path(
        G=G,
        source=start_stop_id,
        target=end_stop_id,
        heuristic=manhattan_heuristic,
        weight="weight"
    )
    
    optimal_duration = nx.astar_path_length(
        G=G,
        source=start_stop_id,
        target=end_stop_id,
        heuristic=manhattan_heuristic,
        weight="weight"
    )
    
    print(f"Path: {optimal_path}")
    print(f"Total Duration: {optimal_duration//60} mins")

except nx.NetworkXNoPath:
    print("No path exists between the specified stops.")
except nx.NodeNotFound as e:
    print(f"Node missing from graph: {e}")

Path: ['450014118', ('450014118', np.int64(134302)), ('450028737', np.int64(134302)), ('450015846', np.int64(134302)), ('450018825', np.int64(134302)), ('450018820', np.int64(134302)), ('450018814', np.int64(134302)), ('450018838', np.int64(134302)), ('450018839', np.int64(134302)), ('450024300', np.int64(134302)), ('450024304', np.int64(134302)), ('450018843', np.int64(134302)), ('450018845', np.int64(134302)), ('450018849', np.int64(134302)), ('450018850', np.int64(134302)), ('450018853', np.int64(134302)), ('450018855', np.int64(134302)), ('450018856', np.int64(134302)), ('450018859', np.int64(134302)), ('450027447', np.int64(134302)), ('450018860', np.int64(134302)), ('450024361', np.int64(134302)), ('450016520', np.int64(134302)), ('450016522', np.int64(134302)), ('450016523', np.int64(134302)), ('450016528', np.int64(134302)), ('450016530', np.int64(134302)), ('450016531', np.int64(134302)), ('450022833', np.int64(134302)), ('450022847', np.int64(134302)), ('450022834', np.int64(

## Draw resulting graph

In [37]:
AVAILABLE_COLORS = [
    'blue', 'red', 'green', 'purple', 'orange', 
    'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'pink'
]

# 1. Segment the path and retain stop IDs
segments = []
current_segment = []
current_route = None

for node in optimal_path:
    if isinstance(node, tuple):
        stop_id = node[0]
        route_id = node[1]
        coord = (G.nodes[node]['lat'], G.nodes[node]['lon'])
        
        if route_id != current_route:
            if current_segment and len(current_segment) > 1:
                segments.append({"route_id": current_route, "stops": current_segment})
            
            current_route = route_id
            current_segment = [{"coord": coord, "stop_id": stop_id}]
        else:
            current_segment.append({"coord": coord, "stop_id": stop_id})

if current_segment and len(current_segment) > 1:
    segments.append({"route_id": current_route, "stops": current_segment})

unique_routes = {seg["route_id"] for seg in segments}
route_colors = {
    route: AVAILABLE_COLORS[i % len(AVAILABLE_COLORS)] 
    for i, route in enumerate(unique_routes)
}

# 2. Initialize the map
start_node = optimal_path[0]
start_lat, start_lon = G.nodes[start_node]['lat'], G.nodes[start_node]['lon']
route_map = folium.Map(location=[start_lat, start_lon], zoom_start=12,  tiles='CartoDB positron')

# 3. Plot segments and node dots
for seg in segments:
    route_id = seg["route_id"]
    color = route_colors[route_id]
    
    # Extract just the coordinates for the PolyLine
    line_coords = [stop["coord"] for stop in seg["stops"]]
    
    # Plot the route line
    folium.PolyLine(
        locations=line_coords,
        color=color,
        weight=6,
        opacity=0.8,
        tooltip=f"Route: {route_id}"
    ).add_to(route_map)
    
    # Plot a dot for each stop on this segment
    for stop in seg["stops"]:
        folium.CircleMarker(
            location=stop["coord"],
            radius=4,
            color=color,
            fill=True,
            fill_color="white",
            fill_opacity=1,
            weight=2,
            tooltip=f"Stop: {stop['stop_id']}"
        ).add_to(route_map)

# 4. Add prominent markers for Start, End, and Transfers
folium.Marker(
    location=(start_lat, start_lon),
    popup="Start",
    icon=folium.Icon(color="green", icon="play")
).add_to(route_map)

end_node = optimal_path[-1]
folium.Marker(
    location=(G.nodes[end_node]['lat'], G.nodes[end_node]['lon']),
    popup="Destination",
    icon=folium.Icon(color="red", icon="stop")
).add_to(route_map)

# Add distinct black transfer dots over the route dots
for seg in segments[1:]:
    transfer_coord = seg["stops"][0]["coord"]
    folium.CircleMarker(
        location=transfer_coord,
        radius=6,
        color="black",
        fill=True,
        fill_color="black",
        fill_opacity=1,
        tooltip="Transfer Point"
    ).add_to(route_map)

route_map.save("a_star_optimal_route.html")